In [7]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Define file paths
file_paths = {
    "social_media": r"C:\Users\iamak\Documents\Applied\Social_Media_Sessions_100K.xlsx",
    "transactions": r"C:\Users\iamak\Documents\Applied\Transaction_100K_Dataset.xlsx",
    "ad_influence": r"C:\Users\iamak\Documents\Applied\Ad_Influence_100K_Dataset (1).xlsx"
}

# Load each dataset (reads the first sheet by default)
transactions_df = pd.read_excel(file_paths["transactions"])
social_df = pd.read_excel(file_paths["social_media"])
ads_df = pd.read_excel(file_paths["ad_influence"])

# Step 2: Convert datetime columns
transactions_df['Transaction_DateTime'] = pd.to_datetime(transactions_df['Transaction_DateTime'])
social_df['Session_End'] = pd.to_datetime(social_df['Session_End'])
social_df['Session_Start'] = pd.to_datetime(social_df['Session_Start'])
ads_df['Ad_Timestamp'] = pd.to_datetime(ads_df['Ad_Timestamp'])
ads_df['Transaction_Timestamp'] = pd.to_datetime(ads_df['Transaction_Timestamp'])

# Step 3: Sort required DataFrames
transactions_sorted = transactions_df.sort_values(by='Transaction_DateTime')
ads_sorted = ads_df.sort_values(by='Ad_Timestamp')
social_sorted = social_df.sort_values(by='Session_End')

# Step 4: Merge ad influence (up to 2 hours before transaction)
merged_ad = pd.merge_asof(
    transactions_sorted,
    ads_sorted,
    by='User_ID',
    left_on='Transaction_DateTime',
    right_on='Ad_Timestamp',
    direction='backward',
    tolerance=pd.Timedelta("2H")
)

# Step 5: Merge social media sessions (session end up to 3 hours before transaction)
merged_full = pd.merge_asof(
    merged_ad.sort_values(by='Transaction_DateTime'),
    social_sorted,
    by='User_ID',
    left_on='Transaction_DateTime',
    right_on='Session_End',
    direction='backward',
    tolerance=pd.Timedelta("3H")
)

# Optional: Check final shape
print("Merged dataset shape:", merged_full.shape)

# Optional: Save to CSV
merged_full.to_csv("merged_dataset.csv", index=False)

# Copy merged dataset to work on
df = merged_full.copy()

# --------------------------
# Rule-Based Scoring Logic
# --------------------------

# 1. Time of Day Score: Evening/Night → 0.3
df['Time_Score'] = df['Time_of_Day_Bucket'].apply(
    lambda x: 0.3 if str(x).lower() in ['evening', 'night'] else 0
)

# 2. Days Since Salary Score: >15 days → 0.2
df['Salary_Score'] = df['Days_Since_Salary'].apply(
    lambda x: 0.2 if pd.notnull(x) and x < 15 else 0
)

# 3. Category Score: Impulse-prone categories → 0.2
impulse_categories = ['Entertainment', 'Fashion', 'Electronics']
df['Category_Score'] = df['Category'].apply(
    lambda x: 0.2 if str(x) in impulse_categories else 0
)

# 4. Ad Influence Score: ad seen within influence window + matched category → 0.2
df['Ad_Influence_Score'] = df.apply(
    lambda row: 0.2 if row.get('Within_Influence_Window') == 1 and row.get('Category_Match') == 1 else 0,
    axis=1
)

# 5. Social Media Session Score: any session before transaction → 0.1
df['Social_Score'] = df['Session_Duration_Min'].apply(
    lambda x: 0.1 if pd.notnull(x) and x > 0 else 0
)

# --------------------------
# Final Impulse Score
# --------------------------
df['Impulse_Score_RuleBased'] = df[
    ['Time_Score', 'Salary_Score', 'Category_Score', 'Ad_Influence_Score', 'Social_Score']
].sum(axis=1)

# Classify as Impulse if score ≥ 0.7
df['Is_Impulse_RuleBased'] = df['Impulse_Score_RuleBased'].apply(
    lambda x: 1 if x >= 0.7 else 0
)

# Optional: View summary
impulse_rate = df['Is_Impulse_RuleBased'].value_counts(normalize=True).round(3)
print("Impulse classification rate:\n", impulse_rate)

# Optional: Save the rule-based data
# df.to_csv("rule_based_scored_transactions.csv", index=False)
# Use the rule-based scored dataset
df_fe = df.copy()

# 1. Extract transaction hour (0–23)
df_fe['Transaction_Hour'] = df_fe['Transaction_DateTime'].dt.hour

# 2. Identify weekday vs weekend
df_fe['Day_Type'] = df_fe['Day_of_Week'].apply(
    lambda x: 'Weekend' if str(x) in ['Saturday', 'Sunday'] else 'Weekday'
)

# 3. Check if ad was seen in the same hour as the transaction
df_fe['Ad_Same_Hour'] = (
    df_fe['Transaction_DateTime'].dt.hour == df_fe['Ad_Timestamp'].dt.hour
).astype(int)

# 4. Check if social session overlapped with transaction
df_fe['Session_Overlap'] = df_fe.apply(
    lambda row: 1 if pd.notnull(row['Session_Start']) and pd.notnull(row['Transaction_DateTime'])
                   and row['Session_Start'] <= row['Transaction_DateTime'] <= row['Session_End'] else 0,
    axis=1
)

# 5. Label Encode 'Category' and 'Platform'
for col in ['Category', 'Platform']:
    if col in df_fe.columns:
        le = LabelEncoder()
        df_fe[col + '_Encoded'] = le.fit_transform(df_fe[col].astype(str))
    else:
        df_fe[col + '_Encoded'] = np.nan  # if column is missing, fill with NaN

C:\Users\iamak\AppData\Local\Temp\ipykernel_24340\834869601.py:38: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  tolerance=pd.Timedelta("2H")
C:\Users\iamak\AppData\Local\Temp\ipykernel_24340\834869601.py:49: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  tolerance=pd.Timedelta("3H")


Merged dataset shape: (100000, 23)
Impulse classification rate:
 Is_Impulse_RuleBased
0    0.969
1    0.031
Name: proportion, dtype: float64
